# 02 - 样本元数据提取

从 GEO SOFT 文件中提取样本级临床信息，包括治疗响应标签。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

base_dir = Path('..')
metadata_dir = base_dir / 'data' / 'metadata'

print(f'元数据目录: {metadata_dir.resolve()}')

## 1. 提取样本元数据

In [ ]:
# 运行提取脚本（需要先下载元数据）
# !python ../scripts/extract_sample_metadata.py --all

# 查看已提取的元数据文件
if metadata_dir.exists():
    meta_files = list(metadata_dir.glob('*_sample_metadata.csv'))
    print(f'已提取的元数据文件: {len(meta_files)}')
    for f in sorted(meta_files):
        df = pd.read_csv(f)
        print(f'  {f.name}: {df.shape[0]} 样本, {df.shape[1]} 字段')
else:
    print('元数据目录不存在，请先运行 download_geo.py')

## 2. 查看临床标签字段

In [ ]:
# 检查训练队列的响应标签
response_keywords = ['response', 'responder', 'outcome', 'treatment', 'therapy', 'remission', 'mucosal']

meta_files = list(metadata_dir.glob('*_sample_metadata.csv')) if metadata_dir.exists() else []
for f in sorted(meta_files)[:3]:
    df = pd.read_csv(f)
    print(f'\n--- {f.name} ---')
    print(f'字段: {list(df.columns)}')
    char_cols = [c for c in df.columns if c.startswith('char_')]
    response_cols = [c for c in char_cols if any(kw in c.lower() for kw in response_keywords)]
    if response_cols:
        print(f'可能的响应标签: {response_cols}')
        for col in response_cols:
            print(f'  {col}: {df[col].value_counts().to_dict()}')
    else:
        print('未自动识别到响应标签')
        print(f'所有characteristics字段: {char_cols}')

## 3. 治疗药物统计

In [ ]:
training = pd.read_csv(base_dir / 'data_manifest' / 'training_cohorts.csv')
print('训练队列治疗药物分布:')
print(training.groupby('treatment')['dataset_id'].apply(list).to_string())
print(f'\n总样本数 (报告值): {training["sample_count_reported"].sum()}')